In [ ]:
from manim import *
import numpy as np
import requests as req
import openai
import time
from tqdm.auto import tqdm
import os

import edge_tts
from pathlib import Path
from mutagen.mp3 import MP3
import random
import string

import moviepy

/home/l0stsky/ExTraStuff/Competitions/LearnAnything/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from SYSTEM_PROMPT_H import *
from SYSTEM_PROMPT_S import *
from SYSTEM_PROMPT_M import *

In [ ]:
VOICE_M='en-US-AndrewNeural'
VOICE_F='en-US-AvaNeural'

LANG='en'


In [ ]:
BASEPATH_M=Path(__file__).resolve().parent
BASEPATH=os.path.join(BASEPATH_M,"temp")

In [ ]:
ML_S=[]

In [ ]:
CLIENT = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="JustSomeRandomKey"
)


In [ ]:
def ContactAI(p,sp,img=None):
    idx =0
    Unc_C=0
    for _ in range(5):
        idx=idx%len(ML_S)
        Model=ML_S[idx]
        
        try:
            response = CLIENT.chat.completions.create(
                model=Model,
                messages=[
                    {"role": "system", "content": sp},
                    {"role": "user", "content": p}
                ])         
            # r_body=response.read().decode('utf-8') 
            # rs=json.loads(r_body)
            # if rs.get('error'):
                # print(rs['error'])
                # ix+=1
                # continue
            ct = response.choices[0].message.content
            print("HELLOOOOO")
            if ct is None:
                print(f"Model {Model} failed, retrying...")
                idx+=1
                time.sleep(1)
                continue
            
            if "Unclear" in ct:
                print("Model Unable to Extract Data, Trying Again")
                if Unc_C>2:
                    return ct
                Unc_C+=1
                idx+=1
                time.sleep(1)
                continue
            return ct
        
        
        except openai.APIError as e:
            print(f"API Error from {Model}: {e}")
            idx += 1
            time.sleep(1)
            
        except Exception as e:
            print(f'Connecting Failed with Error {e}')
            idx +=1
            time.sleep(1)
    else:
        return None


# Layer 1 : Scripter Model

In [ ]:
def scrptGen(USER_INPUT):
    sc=ContactAI(USER_INPUT,SYSTEM_PROMPT_S)
    sects=[i for i in sc["sections"]]

    print(f"SCRIPT Generated!!")
    print(f"SCRIPT Title : {sc['project_title']}")
    print(f"SCRIPT Expected Duration : {sum([i['estimated_duration_seconds'] for i in sc['sections']])}")
    print(f"SCRIPT Num Sects : {len(sects)}")
    
    
    sects.sort(key=lambda i : i["section_number"])
    return sects, sc["project_title"]

sects=scrptGen()

# Layer 2 : Generating VoiceOVER Code

In [ ]:
def voice(sect,n):
    path=os.path.join(BASEPATH,f"Section_{n}")
    sect["path"]=path
    voiceovert=sect["narration_script"]
    sect["voiceoverp"]=os.path.join(sect["path"],voiceover.mp4)
    voiceover = edge_tts.Communicate(voiceovert, VOICE_M, rate="-5%")
    voiceover.save(sect["voiceoverp"])
    sect["voiceoverLen"]=MP3(sect["voiceoverp"]).info.length+2

# Layer 3 : Generating MANIN Code

In [ ]:
from fixManim import *
# import sys
import subprocess

In [ ]:
# def Healer(Code, Model, Tries):
#     prompt=f"This the CODE for a Animation made in MANIMCE, it has the following errors {err}, Fix it"

In [ ]:
def runM(Code, sect, n):
    path=os.path.join(sect["path"],"ManimC.py")
    
    p=os.path.join(sect["path"])
    with open(path, "w") as f:
        f.write(Code)
    try:
        out=subprocess.run(["manim","-ql","-v","ERROR","--media_dir",p,"--custom_folders","-o",f"Section_{n}.mp4",path, f"Section_{n}"], capture_output=True, text=True, check=True)
        return None
    except subprocess.CalledProcessError as e:
        errl=[]
        for line in e.stderr.splitlines():
            if "ERROR" in line or "Exception" in line or line.strip().startswith("File "):
                errl.append(line)
        summ = "\n".join(errl)
        print(f"section {n} Code failed w/ error :\n", summ)
        return summ

In [ ]:
def Heal(Code, sect, pr, n, err, L=0):
    if L>2:
        print(f"The Code Cannot be Healed, Skipping section {n}")
        return 1
    print(f"Healing section {n}, Try : {L+1}")
    
    # TODO : the Err log is huge so idk strip it or smth
    prN=pr+ f"""
    - Animation Code Failed: 
    ```python
    {Code}
    ```
    - Animation Error: 
    ```text
    {err}
    ```
    """
    ManimCode=ContactAI(prN,SYSTEM_PROMPT_H)
    ManimCode=fix_and_validate(ManimCode)
    out=runM(ManimCode,sect,n)
    if out==None:
        print(f"Render Done for Section {n}")
        return 0
    else:
        err=out
        return Heal(ManimCode, sect, pr, n, err, L+1)

In [ ]:
def RenderVid(section, n):
    prompt = f"""
    Render the following scene in ManimCE (AVOID USING MANIMGL):
    - Topic: {section['topic']} ({section['subtopic']})
    - Target Duration: {section['voiceoverLen']} seconds
    - Narration Script (for pacing): "{section['narration_script']}"
    - Animation Vision: "{section['animation_vision']}"
    """
    ManimCode=ContactAI(prompt,SYSTEM_PROMPT_M)
    ManimCode=fix_and_validate(ManimCode)
    out=runM(ManimCode,section,n)
    if out==None:
        print(f"Render Done for Section {n}")
    else:
        Heal(ManimCode,section,prompt,n,out)

In [ ]:
def Mergevids(sects,title):
    vids=[]
    for n,sect in enumerate(sects):
        path=os.path.join(sect["path"],f"Section_{n}.mp4")
        vids.append(moviepy.editor.VideoFileClip(path))
    final=moviepy.editor.concatenate_videoclips(vids,method='compose')
    
    final.write_videofile(os.path.join((os.path.join((os.path.join(BASEPATH_M,"output")),f"{title}")),"out.mp4"))
    
    for v in vids : v.close()

In [ ]:
import threading
def Main(USER_INPUT):
    sects,title=scrptGen(USER_INPUT)
    ths=[]
    for n,sect in enumerate(sects):
        t=threading.Thread(target=voice, args=(sect,n))
        ths.append(t)
    for t in ths:
        t.start()
    for t in ths:
        t.join()
    ths=[]
    for n,sect in enumerate(sects):
        t=threading.Thread(target=RenderVid, args=(sect,n))
        ths.append(t)
    for t in ths:
        t.start()
    for t in ths:
        t.join()
    Mergevids(sects,title)
    

In [ ]:
# TODO : Merge the Vids => Done
# TODO : Verify Video By sending Screenshots => Mght not be needed
# TODO : Strip the Error logs => Done!!

# Testing

In [ ]:
USER_INPUT="Tell me about Neural Networks. Like Idk How Do they Even LEARN ANYTHING? Arent they Just Math??? Also Wtf are those Graph like things with a name with smth like a network."

In [ ]:
Main(USER_INPUT)